# Simulator Benchmark: SERGIO vs qSimCells vs scMultiSim

A side-by-side benchmark of three single-cell RNA-seq simulators on an identical
five-gene cascade scenario (mono-culture and co-culture).

| Simulator | Approach | Language |
|-----------|----------|----------|
| SERGIO | Langevin SDE + Hill kinetics | Python |
| qSimCells | Quantum PQC + Born-rule sampling + NB | Python / Qiskit |
| scMultiSim | CIF hierarchy + kinetic rates | R (via subprocess) |

**Ground-truth GRN:** G0 → G1 → G2 → G3 (cascade), G4 independent  
**Conditions:** mono-culture (Type A only) and co-culture (Type A + Type B mixed)

## 0  Dependencies

In [ ]:
# Core scientific stack (already in qsim_cells_env)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import scanpy as sc
import anndata as ad
import warnings, os, sys, subprocess
warnings.filterwarnings("ignore")

from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, roc_auc_score
from sklearn.preprocessing import StandardScaler
from scipy.stats import nbinom, ks_2samp, pearsonr, spearmanr
from scipy.optimize import minimize
from scipy.special import digamma

# arboreto GRN inference
from arboreto.algo import grnboost2, genie3

# qSimCells generative module (same repo)
REPO_ROOT = os.path.abspath("..")
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
from qsim_cells.generative import (
    create_rotation_circuit,
    concatenate_circuits_with_separate_measurements,
    add_crx_and_measurements_to_circuit,
    create_binary_matrix,
    create_count_matrix_nbinom,
)
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime import SamplerV2 as Sampler

print("All imports OK.")

## 1  Shared scenario parameters

In [ ]:
# ── Gene / circuit layout ──────────────────────────────────────────────────
N_GENES    = 5          # qubits / simulated genes
N_CELLS    = 500        # cells per cell type
SEED       = 42
rng        = np.random.default_rng(SEED)

GENE_NAMES = [f"G{i}" for i in range(N_GENES)]

# ── Ground-truth directed GRN ──────────────────────────────────────────────
# Cascade:  G0 → G1 → G2 → G3      G4 independent (control)
GT_EDGES = [(0, 1), (1, 2), (2, 3)]          # (regulator, target)
GT_ADJ   = np.zeros((N_GENES, N_GENES))       # GT_ADJ[reg, target] = 1
for (r, t) in GT_EDGES:
    GT_ADJ[r, t] = 1.0

# ── NB augmentation params (shared by qSimCells; reference for others) ──
MU_VEC = np.array([50., 40., 35., 30., 20.])  # mean expression per gene (ON)
R_VEC  = np.array([ 5.,  5.,  5.,  5.,  8.])  # dispersion parameter

# Angles for qSimCells cell types
# Type A (cascade active): high Ry on driver genes
ANGLES_A = np.array([0.90, 0.75, 0.70, 0.68, 0.35]) * np.pi
# Type B (cascade inactive): low driver, different profile
ANGLES_B = np.array([0.15, 0.20, 0.20, 0.20, 0.65]) * np.pi

print(f"Scenario: {N_GENES} genes, {N_CELLS} cells/type, seed={SEED}")
print(f"GT directed edges: {GT_EDGES}")

## 2  SERGIO simulation

Self-contained Langevin-SDE implementation following Dibaeinia & Sinha (2020).
Each gene follows:

    dX_i/dt = prod_i(X) − λ · X_i + σ · ξ(t)

where `prod_i` is the sum of Hill-function contributions from regulators,
master regulators have a constant bin-specific basal production rate,
and technical noise is applied as Poisson sampling at read-out.

In [ ]:
class SERGIO:
    """
    Minimal faithful reproduction of SERGIO Langevin dynamics.
    Reference: Dibaeinia & Sinha, Cell Systems 2020.
    """

    def __init__(self, n_genes, n_bins, n_cells,
                 noise=0.1, decay=0.8, dt=0.005,
                 n_burnin=3000, n_sample_steps=15):
        self.n_genes        = n_genes
        self.n_bins         = n_bins
        self.n_cells        = n_cells
        self.noise          = noise
        self.decay          = decay
        self.dt             = dt
        self.n_burnin       = n_burnin
        self.n_sample_steps = n_sample_steps
        self._master_regs   = {}   # gene_id -> [basal_rate_bin0, ...]
        self._targets       = {}   # gene_id -> [(reg_id, K, hill), ...]

    def add_master_regulator(self, gene_id, basal_rates):
        """basal_rates: list of length n_bins"""
        self._master_regs[gene_id] = np.array(basal_rates, dtype=float)

    def add_target(self, gene_id, regulators):
        """regulators: list of (reg_id, K, hill_coeff)"""
        self._targets[gene_id] = regulators

    @staticmethod
    def _hill(x, K, h):
        xh = x ** h
        return xh / (K ** h + xh)

    def _prod(self, x, gene_id, bin_id):
        if gene_id in self._master_regs:
            return self._master_regs[gene_id][bin_id]
        regs = self._targets.get(gene_id, [])
        return sum(K * self._hill(max(x[rid], 0.), K, h) for rid, K, h in regs)

    def simulate(self, seed=42):
        rng_loc = np.random.default_rng(seed)
        self.expressions = []           # list of (n_cells, n_genes) arrays

        for b in range(self.n_bins):
            cells = []
            # initialise all cells at rough steady state of master regs
            x0 = np.array([
                self._master_regs.get(g, {}).get(b, 0.5) / self.decay
                if isinstance(self._master_regs.get(g, None), np.ndarray)
                else 0.5
                for g in range(self.n_genes)
            ])

            for c in range(self.n_cells):
                x = x0.copy() + rng_loc.uniform(-0.1, 0.1, self.n_genes)
                x = np.maximum(x, 0.)

                # Burn-in (reach local steady state)
                for _ in range(self.n_burnin):
                    prod = np.array([self._prod(x, g, b) for g in range(self.n_genes)])
                    dx   = (prod - self.decay * x) * self.dt
                    dx  += self.noise * rng_loc.standard_normal(self.n_genes) * np.sqrt(self.dt)
                    x    = np.maximum(x + dx, 0.)

                # Sample n_sample_steps states and average (reduces shot noise)
                samples = []
                for _ in range(self.n_sample_steps):
                    prod = np.array([self._prod(x, g, b) for g in range(self.n_genes)])
                    dx   = (prod - self.decay * x) * self.dt
                    dx  += self.noise * rng_loc.standard_normal(self.n_genes) * np.sqrt(self.dt)
                    x    = np.maximum(x + dx, 0.)
                    samples.append(x.copy())

                cells.append(np.mean(samples, axis=0))

            self.expressions.append(np.array(cells))   # (n_cells, n_genes)

    def to_counts(self, umi_scale=50, seed=42):
        """Poisson-downsample to UMI-like integer counts."""
        rng_loc = np.random.default_rng(seed)
        expr = np.vstack(self.expressions)        # (n_bins*n_cells, n_genes)
        return rng_loc.poisson(expr * umi_scale).astype(np.int32)

print("SERGIO class defined.")

In [ ]:
# ── Build SERGIO GRN ───────────────────────────────────────────────────────
# Cascade: G0 (master reg) → G1 → G2 → G3,  G4 independent master reg
#
# Type A (bin 0): G0 basal = 2.0 (high) → cascade fully active
# Type B (bin 1): G0 basal = 0.1 (low)  → cascade largely silent

sim_sergio = SERGIO(n_genes=N_GENES, n_bins=2, n_cells=N_CELLS,
                    noise=0.08, decay=0.8, dt=0.005,
                    n_burnin=2000, n_sample_steps=15)

# Master regulators
sim_sergio.add_master_regulator(0, [2.0, 0.12])   # G0: driver
sim_sergio.add_master_regulator(4, [0.6,  0.5 ])   # G4: independent control

# Target cascade   (reg_id, K, hill)
sim_sergio.add_target(1, [(0, 1.5, 2)])
sim_sergio.add_target(2, [(1, 1.5, 2)])
sim_sergio.add_target(3, [(2, 1.5, 2)])

# ── Simulate ──────────────────────────────────────────────────────────────
np.random.seed(SEED)
sim_sergio.simulate(seed=SEED)

counts_sergio = sim_sergio.to_counts(umi_scale=25, seed=SEED)
labels_sergio = np.array(["TypeA"] * N_CELLS + ["TypeB"] * N_CELLS)

# ── mono (Type A only) / co-culture (both) ────────────────────────────────
counts_sergio_mono = counts_sergio[:N_CELLS]
counts_sergio_co   = counts_sergio                # both types

print(f"SERGIO counts shape: {counts_sergio_co.shape}")
print(f"SERGIO G0 mean TypeA={counts_sergio[:N_CELLS, 0].mean():.1f}  TypeB={counts_sergio[N_CELLS:, 0].mean():.1f}")
print(f"SERGIO G1 mean TypeA={counts_sergio[:N_CELLS, 1].mean():.1f}  TypeB={counts_sergio[N_CELLS:, 1].mean():.1f}")

## 3  qSimCells simulation

Uses the PQC from `generative.py`:
- **Ry(θ)** on each qubit encodes gene activation probability  
- **CX cascade** (G0→G1→G2→G3) encodes regulatory dependency  
- **NB augmentation** adds overdispersed counts  

Two cell types share the same circuit architecture with different Ry angles.

In [ ]:
def run_qsimcells(angles_a, angles_b, n_shots_per_type, mu_vec, r_vec,
                  interaction_map, seed):
    """
    Simulate qSimCells co-culture (Type A + Type B) and return
    (count_matrix, cell_type_labels).
    """
    backend = AerSimulator(seed_simulator=seed)

    # Build circuits
    circ_a = create_rotation_circuit(angles_a)
    circ_b = create_rotation_circuit(angles_b)
    combined = concatenate_circuits_with_separate_measurements(circ_a, circ_b)
    circuit  = add_crx_and_measurements_to_circuit(
        combined, circ1_num_qubits=len(angles_a),
        interaction_map=interaction_map, crx_angle=np.pi)

    # Run
    from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
    pm   = generate_preset_pass_manager(backend=backend, optimization_level=1)
    qc   = pm.run(circuit)
    job  = Sampler(mode=backend).run([qc], shots=n_shots_per_type)
    res  = job.result()[0]

    counts_a = res.data.c_measure1.get_counts()
    counts_b = res.data.c_measure2.get_counts()

    bin_a = create_binary_matrix(counts_a)   # (shots_a, n_genes)
    bin_b = create_binary_matrix(counts_b)   # (shots_b, n_genes)

    np.random.seed(seed)
    cnt_a = create_count_matrix_nbinom(bin_a, mu_vec, r_vec)
    cnt_b = create_count_matrix_nbinom(bin_b, mu_vec, r_vec)

    counts = np.vstack([cnt_a, cnt_b])
    labels = np.array(["TypeA"] * len(bin_a) + ["TypeB"] * len(bin_b))
    return counts, labels

# CX cascade qubits: 0→1, 1→2, 2→3 (G4 qubit 4 is independent — no CX)
INTERACTION_MAP = [(0, 1), (1, 2), (2, 3)]

counts_qsim, labels_qsim = run_qsimcells(
    ANGLES_A, ANGLES_B,
    n_shots_per_type=N_CELLS,
    mu_vec=MU_VEC, r_vec=R_VEC,
    interaction_map=INTERACTION_MAP,
    seed=SEED,
)

counts_qsim_mono = counts_qsim[labels_qsim == "TypeA"]
counts_qsim_co   = counts_qsim

print(f"qSimCells counts shape: {counts_qsim_co.shape}")
print(f"qSimCells G0 mean TypeA={counts_qsim[labels_qsim=='TypeA', 0].mean():.1f}  TypeB={counts_qsim[labels_qsim=='TypeB', 0].mean():.1f}")
print(f"qSimCells G1 mean TypeA={counts_qsim[labels_qsim=='TypeA', 1].mean():.1f}  TypeB={counts_qsim[labels_qsim=='TypeB', 1].mean():.1f}")

## 4  scMultiSim simulation

scMultiSim uses Cell Identity Factors (CIFs) and kinetic parameters
(burst frequency β, burst size α) modulated by the GRN.

**Option A** — if R + scMultiSim are available, the cell below writes and runs
an R script, then loads the CSV outputs.

**Option B** — otherwise the cell falls back to a self-contained Python
CIF-based reproduction capturing the same statistical structure.

In [ ]:
R_SCRIPT = """
suppressPackageStartupMessages({
  if (!requireNamespace('scMultiSim', quietly=TRUE))
    remotes::install_github('ZhangLabGT/scMultiSim', quiet=TRUE)
  library(scMultiSim)
  library(ape)
})

set.seed(42)
n_cells_per_type <- 500

# GRN: cascade G1->G2->G3->G4, G5 independent (1-indexed in R)
grn <- data.frame(
  regulator = c(1L, 2L, 3L),
  target    = c(2L, 3L, 4L),
  effect    = c(1.0, 1.0, 1.0)
)

# -- Co-culture: two cell types via bifurcating tree
tree_co <- read.tree(text='((TypeA:1,TypeB:1):0);')
opts_co  <- list(
  GRN               = grn,
  num.cells         = n_cells_per_type * 2,
  num.cifs          = 20L,
  tree              = tree_co,
  diff.cif.fraction = 0.8,
  sigma.b           = 0.4,
  scale.s           = 1.0
)
res_co    <- sim_true_counts(opts_co)
counts_co <- t(res_co$counts)                  # cells x genes
meta_co   <- res_co$cell_meta

# -- Mono-culture: one cell type (flat star tree)
tree_mono <- read.tree(text='(TypeA:1);')
opts_mono <- list(
  GRN               = grn,
  num.cells         = n_cells_per_type,
  num.cifs          = 20L,
  tree              = tree_mono,
  diff.cif.fraction = 0.0,
  sigma.b           = 0.2,
  scale.s           = 1.0
)
res_mono    <- sim_true_counts(opts_mono)
counts_mono <- t(res_mono$counts)

# Save
write.csv(counts_co,   'scmultisim_co.csv',   row.names=FALSE)
write.csv(counts_mono, 'scmultisim_mono.csv', row.names=FALSE)
write.csv(meta_co,     'scmultisim_meta.csv', row.names=FALSE)
cat('scMultiSim: done\n')
"""

SCRIPT_PATH  = "scmultisim_run.R"
SCM_CO_PATH  = "scmultisim_co.csv"
SCMULTISIM_AVAILABLE = False

with open(SCRIPT_PATH, "w") as fh:
    fh.write(R_SCRIPT)

try:
    result = subprocess.run(
        ["Rscript", SCRIPT_PATH],
        capture_output=True, text=True, timeout=120
    )
    if result.returncode == 0 and os.path.exists(SCM_CO_PATH):
        SCMULTISIM_AVAILABLE = True
        print("scMultiSim R run succeeded.")
    else:
        print("R run failed or timed out — using Python fallback.")
        if result.stderr:
            print(result.stderr[:500])
except Exception as e:
    print(f"R not available ({e}) — using Python fallback.")

In [ ]:
if SCMULTISIM_AVAILABLE:
    scm_co   = pd.read_csv("scmultisim_co.csv").values.astype(np.int32)
    scm_mono = pd.read_csv("scmultisim_mono.csv").values.astype(np.int32)
    scm_meta = pd.read_csv("scmultisim_meta.csv")
    n_a = n_b = N_CELLS
    labels_scm = np.array(["TypeA"] * n_a + ["TypeB"] * n_b)
else:
    # ── Python CIF fallback ───────────────────────────────────────────────
    # Cell Identity Factors: each cell type has a distinct CIF distribution.
    # Gene expression ~ Poisson(burst_freq * burst_size),
    # where burst_freq is modulated by CIFs + GRN cascade.

    def simulate_cif(n_cells, n_genes, grn_adj, type_labels, n_cifs=20,
                     scale=40, seed=42):
        rng_l = np.random.default_rng(seed)
        n_types = len(np.unique(type_labels))

        # CIF means differ by cell type (captures diff.cif.fraction)
        cif_means = rng_l.standard_normal((n_types, n_cifs)) * 1.5
        cifs = np.zeros((n_cells, n_cifs))
        type_ids = np.unique(type_labels)
        for ti, tname in enumerate(type_ids):
            mask = type_labels == tname
            cifs[mask] = rng_l.standard_normal((mask.sum(), n_cifs)) * 0.4 + cif_means[ti]

        # Gene loadings  (n_cifs, n_genes)
        loadings = rng_l.standard_normal((n_cifs, n_genes)) * 0.5
        mu_base  = np.maximum(cifs @ loadings, 0.)   # (n_cells, n_genes)

        # GRN cascade: propagate activations (3 passes)
        mu = mu_base.copy()
        for _ in range(3):
            mu += 0.35 * np.maximum(mu @ grn_adj, 0.)   # reg -> target boost

        # Kinetic model: Poisson(scale * softplus(mu))
        lambda_mat = scale * np.log1p(np.exp(mu))
        counts = rng_l.poisson(lambda_mat).astype(np.int32)
        return counts

    all_labels_scm = np.array(["TypeA"] * N_CELLS + ["TypeB"] * N_CELLS)
    scm_co   = simulate_cif(N_CELLS * 2, N_GENES, GT_ADJ, all_labels_scm,
                            n_cifs=20, scale=35, seed=SEED)
    scm_mono = simulate_cif(N_CELLS, N_GENES, GT_ADJ,
                            np.array(["TypeA"] * N_CELLS),
                            n_cifs=20, scale=35, seed=SEED)
    labels_scm = all_labels_scm
    print("scMultiSim (Python CIF fallback) simulated.")

counts_scm_co   = scm_co
counts_scm_mono = scm_mono
print(f"scMultiSim counts shape: {counts_scm_co.shape}")

## 5  Build AnnData objects

In [ ]:
def make_adata(counts, labels, sim_name, condition):
    adata = ad.AnnData(
        X   = counts.astype(np.float32),
        obs = pd.DataFrame({
            "cell_type": labels,
            "simulator": sim_name,
            "condition": condition,
        }),
        var = pd.DataFrame(index=GENE_NAMES),
    )
    adata.obs_names = [f"{sim_name}_{condition}_{i}" for i in range(len(labels))]
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    return adata

# Co-culture AnnData
adata_sergio = make_adata(counts_sergio_co, labels_sergio, "SERGIO",   "co")
adata_qsim   = make_adata(counts_qsim_co,   labels_qsim,   "qSimCells","co")
adata_scm    = make_adata(counts_scm_co,    labels_scm,    "scMultiSim","co")

adatas_co = {"SERGIO": adata_sergio, "qSimCells": adata_qsim, "scMultiSim": adata_scm}

# Raw count AnnData for NB fitting
raw_co = {
    "SERGIO":    counts_sergio_co,
    "qSimCells": counts_qsim_co,
    "scMultiSim":counts_scm_co,
}
labels_co = {
    "SERGIO":    labels_sergio,
    "qSimCells": labels_qsim,
    "scMultiSim":labels_scm,
}

print("AnnData objects built.")
for name, ad_ in adatas_co.items():
    print(f"  {name}: {ad_.shape}  types={ad_.obs['cell_type'].value_counts().to_dict()}")

## 6  Cell type separation — PCA & UMAP

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
PALETTE = {"TypeA": "#2196F3", "TypeB": "#FF5722"}
sil_scores = {}

for col, (name, adata) in enumerate(adatas_co.items()):
    X_raw = adata.X if not hasattr(adata.X, 'toarray') else adata.X.toarray()

    # PCA
    pca   = PCA(n_components=10, random_state=SEED)
    X_pca = pca.fit_transform(X_raw)
    ax    = axes[0, col]
    for ct, color in PALETTE.items():
        mask = adata.obs["cell_type"].values == ct
        ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
                   c=color, alpha=0.45, s=12, label=ct, rasterized=True)
    ax.set_title(f"{name}\nPCA", fontsize=12, fontweight="bold")
    ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
    ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
    if col == 0:
        ax.legend(frameon=False, fontsize=9)

    sil = silhouette_score(X_pca, adata.obs["cell_type"].values)
    sil_scores[name] = sil

    # UMAP
    sc.pp.neighbors(adata, n_pcs=10, random_state=SEED)
    sc.tl.umap(adata, random_state=SEED)
    ax2 = axes[1, col]
    for ct, color in PALETTE.items():
        mask = adata.obs["cell_type"].values == ct
        ax2.scatter(adata.obsm["X_umap"][mask, 0], adata.obsm["X_umap"][mask, 1],
                    c=color, alpha=0.45, s=12, rasterized=True)
    ax2.set_title(f"{name}\nUMAP (sil={sil:.3f})", fontsize=11)
    ax2.set_xlabel("UMAP 1"); ax2.set_ylabel("UMAP 2")

plt.tight_layout()
plt.savefig("benchmark_pca_umap.pdf", bbox_inches="tight")
plt.show()
print("Silhouette scores (PCA):", sil_scores)

## 7  Gene expression distributions

In [ ]:
fig, axes = plt.subplots(N_GENES, 3, figsize=(14, 3 * N_GENES), sharey=False)
COLORS_SIM = {"SERGIO": "#4CAF50", "qSimCells": "#9C27B0", "scMultiSim": "#FF9800"}

for col, (sim_name, counts) in enumerate(raw_co.items()):
    labs = labels_co[sim_name]
    for row, gname in enumerate(GENE_NAMES):
        ax = axes[row, col]
        for ct, color in PALETTE.items():
            vals = counts[labs == ct, row]
            ax.hist(vals, bins=40, alpha=0.55, color=color,
                    label=ct, density=True, histtype="stepfilled")
        ax.set_title(f"{sim_name} — {gname}", fontsize=9)
        ax.set_xlabel("UMI count", fontsize=8)
        if col == 0:
            ax.set_ylabel("Density", fontsize=8)
        if row == 0 and col == 0:
            ax.legend(fontsize=8, frameon=False)

plt.tight_layout()
plt.savefig("benchmark_distributions.pdf", bbox_inches="tight")
plt.show()

## 8  Mean-variance relationship

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

for sim_name, counts in raw_co.items():
    # use TypeA cells only for consistency
    labs = labels_co[sim_name]
    cnt  = counts[labs == "TypeA"]
    means = cnt.mean(axis=0)
    vars_ = cnt.var(axis=0)
    ax.scatter(means, vars_, label=sim_name,
               color=COLORS_SIM[sim_name], s=80, alpha=0.85, zorder=3)

# Reference lines: Poisson (var=mean) and NB power law
x_ref = np.linspace(1, max(means) * 1.1, 300)
ax.plot(x_ref, x_ref,          "k--", lw=1.2, label="Poisson (var=mean)")
ax.plot(x_ref, x_ref + x_ref**2 / 5, "k:",  lw=1.2, label="NB r=5")

for sim_name, counts in raw_co.items():
    labs = labels_co[sim_name]
    cnt  = counts[labs == "TypeA"]
    for gi, gname in enumerate(GENE_NAMES):
        ax.annotate(gname, (cnt[:, gi].mean(), cnt[:, gi].var()),
                    fontsize=7, color=COLORS_SIM[sim_name], alpha=0.7)

ax.set_xlabel("Mean expression (TypeA)", fontsize=12)
ax.set_ylabel("Variance", fontsize=12)
ax.set_title("Mean–variance relationship", fontsize=13)
ax.legend(fontsize=9, frameon=False)
ax.set_xscale("log"); ax.set_yscale("log")
plt.tight_layout()
plt.savefig("benchmark_mean_variance.pdf", bbox_inches="tight")
plt.show()

## 9  Zero inflation

In [ ]:
zi_results = {}
for sim_name, counts in raw_co.items():
    zi_results[sim_name] = (counts == 0).mean(axis=0)   # fraction zeros per gene

zi_df = pd.DataFrame(zi_results, index=GENE_NAMES)
print("Zero-inflation (fraction of zeros per gene):")
print(zi_df.round(3))

fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(N_GENES)
w = 0.25
for i, (sim_name, zi) in enumerate(zi_results.items()):
    ax.bar(x + (i - 1) * w, zi, width=w,
           color=COLORS_SIM[sim_name], label=sim_name, alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(GENE_NAMES)
ax.set_ylabel("Fraction zeros", fontsize=11)
ax.set_title("Zero inflation per gene (all cells)", fontsize=12)
ax.legend(fontsize=9, frameon=False)
plt.tight_layout()
plt.savefig("benchmark_zero_inflation.pdf", bbox_inches="tight")
plt.show()

## 10  Negative-Binomial goodness of fit

In [ ]:
def fit_nb_mle(x):
    """Fit NB(r, p) by MLE; return (r, p, log-likelihood)."""
    x = x[x > 0].astype(float)
    if len(x) < 20:
        return None, None, np.nan
    mu_hat = x.mean()
    v_hat  = x.var()
    r0 = mu_hat ** 2 / max(v_hat - mu_hat, 0.1)
    r0 = np.clip(r0, 0.1, 200.)

    def neg_ll(log_r):
        r = np.exp(log_r)
        p = r / (mu_hat + r)
        return -np.sum(nbinom.logpmf(x.astype(int), r, p))

    from scipy.optimize import minimize_scalar
    res = minimize_scalar(neg_ll, bounds=(np.log(0.01), np.log(500)), method="bounded")
    r   = np.exp(res.x)
    p   = r / (mu_hat + r)
    ll  = -res.fun / len(x)    # per-observation log-likelihood
    return r, p, ll

nb_fit = {}
for sim_name, counts in raw_co.items():
    gene_lls = []
    for gi in range(N_GENES):
        _, _, ll = fit_nb_mle(counts[:, gi])
        gene_lls.append(ll)
    nb_fit[sim_name] = gene_lls

nb_df = pd.DataFrame(nb_fit, index=GENE_NAMES)
print("NB per-observation log-likelihood (higher = better NB fit):")
print(nb_df.round(4))

fig, ax = plt.subplots(figsize=(7, 4))
for i, (sim_name, lls) in enumerate(nb_fit.items()):
    ax.bar(x + (i - 1) * w, lls, width=w,
           color=COLORS_SIM[sim_name], label=sim_name, alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(GENE_NAMES)
ax.set_ylabel("Per-obs log-likelihood", fontsize=11)
ax.set_title("NB goodness of fit (higher = better)", fontsize=12)
ax.legend(fontsize=9, frameon=False)
plt.tight_layout()
plt.savefig("benchmark_nb_fit.pdf", bbox_inches="tight")
plt.show()

## 11  GRN recovery — AUROC

GENIE3 is run on TypeA cells from each simulator.  
AUROC is computed by treating every ordered gene pair (i→j, i≠j) as a
binary classification problem: 1 if the pair is a ground-truth edge, else 0.

In [ ]:
def compute_grn_auroc(counts, labels, gt_adj, gene_names, seed, n_estimators=500):
    """
    Run GENIE3 on TypeA cells, return AUROC vs ground-truth directed adjacency.
    """
    cnt_a  = counts[labels == "TypeA"]
    expr_df = pd.DataFrame(cnt_a.astype(float), columns=gene_names)

    # GENIE3 returns a DataFrame (regulator, target, importance)
    df_g3 = genie3(expression_data=expr_df, tf_names=gene_names,
                   verbose=False, seed=seed)

    # Build score matrix
    score_mat = np.zeros((N_GENES, N_GENES))
    gene_idx  = {g: i for i, g in enumerate(gene_names)}
    for _, row in df_g3.iterrows():
        ri = gene_idx[row["TF"]]
        ti = gene_idx[row["target"]]
        score_mat[ri, ti] = row["importance"]

    # Normalise
    if score_mat.max() > 0:
        score_mat /= score_mat.max()

    # Flatten (exclude diagonal)
    mask   = ~np.eye(N_GENES, dtype=bool)
    y_true = gt_adj[mask].astype(int)
    y_score= score_mat[mask]

    auroc  = roc_auc_score(y_true, y_score)
    return auroc, score_mat, df_g3

print("Running GENIE3 on all three simulators (TypeA cells)…")
auroc_results = {}
score_mats    = {}
for sim_name, counts in raw_co.items():
    labs = labels_co[sim_name]
    auroc, smat, _ = compute_grn_auroc(counts, labs, GT_ADJ, GENE_NAMES, seed=SEED)
    auroc_results[sim_name] = auroc
    score_mats[sim_name]    = smat
    print(f"  {sim_name}: AUROC = {auroc:.3f}")

In [ ]:
# ── Plot AUROC bar + importance heatmaps ──────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

# Bar chart
ax = axes[0]
sims   = list(auroc_results.keys())
aucs   = [auroc_results[s] for s in sims]
colors = [COLORS_SIM[s] for s in sims]
bars   = ax.bar(sims, aucs, color=colors, alpha=0.85, width=0.5)
ax.axhline(0.5, color="k", ls="--", lw=1.2, label="Random (0.5)")
ax.set_ylim(0, 1.05)
ax.set_ylabel("AUROC", fontsize=11)
ax.set_title("GRN recovery\n(GENIE3, TypeA)", fontsize=11)
for bar, auc in zip(bars, aucs):
    ax.text(bar.get_x() + bar.get_width()/2, auc + 0.02,
            f"{auc:.3f}", ha="center", fontsize=10, fontweight="bold")
ax.legend(fontsize=9, frameon=False)

# Heatmaps
for ci, sim_name in enumerate(sims):
    ax2 = axes[ci + 1]
    sns.heatmap(score_mats[sim_name], ax=ax2, cmap="Blues",
                xticklabels=GENE_NAMES, yticklabels=GENE_NAMES,
                annot=True, fmt=".2f", cbar=False,
                linewidths=0.5, vmin=0, vmax=1)
    # Mark ground-truth edges
    for (r, t) in GT_EDGES:
        ax2.add_patch(plt.Rectangle((t, r), 1, 1, fill=False,
                                    edgecolor="red", lw=2.5))
    ax2.set_title(f"{sim_name}\nGENIE3 importance (red=GT)", fontsize=10)
    ax2.set_xlabel("Target"); ax2.set_ylabel("Regulator")

plt.tight_layout()
plt.savefig("benchmark_grn_auroc.pdf", bbox_inches="tight")
plt.show()

## 12  Gene–gene Pearson correlation vs ground-truth structure

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ci, (sim_name, counts) in enumerate(raw_co.items()):
    ax  = axes[ci]
    cnt = counts[labels_co[sim_name] == "TypeA"].astype(float)
    corr_mat = np.corrcoef(cnt.T)   # (n_genes, n_genes)
    sns.heatmap(corr_mat, ax=ax, cmap="RdBu_r", center=0,
                xticklabels=GENE_NAMES, yticklabels=GENE_NAMES,
                annot=True, fmt=".2f", cbar=(ci==2),
                linewidths=0.4, vmin=-1, vmax=1)
    # Highlight GT edges
    for (r, t) in GT_EDGES:
        ax.add_patch(plt.Rectangle((t, r), 1, 1, fill=False,
                                   edgecolor="lime", lw=2.5))
        ax.add_patch(plt.Rectangle((r, t), 1, 1, fill=False,
                                   edgecolor="lime", lw=2.5, ls="--"))
    ax.set_title(f"{sim_name}\nPearson correlation (TypeA)", fontsize=10)

plt.tight_layout()
plt.savefig("benchmark_correlation.pdf", bbox_inches="tight")
plt.show()

## 13  Summary benchmark table

In [ ]:
summary = {}
for sim_name in ["SERGIO", "qSimCells", "scMultiSim"]:
    counts = raw_co[sim_name]
    labs   = labels_co[sim_name]

    # Mean zero inflation (all cells)
    zi_mean = (counts == 0).mean()

    # NB fit mean log-likelihood (all genes)
    lls = [fit_nb_mle(counts[:, gi])[2] for gi in range(N_GENES)]
    nb_mean_ll = np.nanmean(lls)

    # Mean-variance slope in log-log (overdispersion indicator; >1 = overdispersed)
    mu_  = counts[labs == "TypeA"].mean(axis=0)
    var_ = counts[labs == "TypeA"].var(axis=0)
    mask_ = (mu_ > 0) & (var_ > 0)
    slope, _ = np.polyfit(np.log(mu_[mask_]), np.log(var_[mask_]), 1)

    summary[sim_name] = {
        "Silhouette (PCA)": round(sil_scores[sim_name], 3),
        "GRN AUROC (GENIE3)": round(auroc_results[sim_name], 3),
        "NB log-lik (mean)": round(nb_mean_ll, 4),
        "Zero inflation": round(zi_mean, 3),
        "Mean-var slope": round(slope, 2),
    }

df_summary = pd.DataFrame(summary).T
print(df_summary.to_string())
df_summary

## 14  Master comparison figure

In [ ]:
fig = plt.figure(figsize=(18, 12))
gs  = gridspec.GridSpec(3, 4, figure=fig, hspace=0.45, wspace=0.38)

# Row 0: PCA plots
for ci, sim_name in enumerate(["SERGIO", "qSimCells", "scMultiSim"]):
    ax = fig.add_subplot(gs[0, ci])
    adata = adatas_co[sim_name]
    X_raw = adata.X if not hasattr(adata.X, 'toarray') else adata.X.toarray()
    X_pca = PCA(n_components=2, random_state=SEED).fit_transform(X_raw)
    for ct, color in PALETTE.items():
        mask = adata.obs["cell_type"].values == ct
        ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
                   c=color, alpha=0.4, s=8, label=ct, rasterized=True)
    ax.set_title(sim_name, fontsize=12, fontweight="bold")
    ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
    if ci == 0:
        ax.legend(fontsize=8, frameon=False, markerscale=2)

# Row 0 col 3: Silhouette scores
ax = fig.add_subplot(gs[0, 3])
sims_ = list(sil_scores.keys())
vals_ = [sil_scores[s] for s in sims_]
ax.barh(sims_, vals_, color=[COLORS_SIM[s] for s in sims_], alpha=0.85)
ax.set_xlabel("Silhouette score"); ax.set_title("Cell-type separability")
ax.set_xlim(0, 1)

# Row 1: Gene expr distributions (G0 and G1 per simulator)
for ci, sim_name in enumerate(["SERGIO", "qSimCells", "scMultiSim"]):
    ax = fig.add_subplot(gs[1, ci])
    counts = raw_co[sim_name]
    labs   = labels_co[sim_name]
    for gi, lty in zip([0, 1], ["-", "--"]):
        for ct, color in PALETTE.items():
            vals = counts[labs == ct, gi]
            ax.hist(vals, bins=30, alpha=0.5, color=color,
                    density=True, histtype="step",
                    lw=1.8, linestyle=lty,
                    label=f"{ct} G{gi}" if ci == 0 else "")
    ax.set_title(f"{sim_name}\nG0 (─) / G1 (--)", fontsize=10)
    ax.set_xlabel("UMI")
    if ci == 0:
        ax.legend(fontsize=7, frameon=False)

# Row 1 col 3: NB fit log-likelihood
ax = fig.add_subplot(gs[1, 3])
nb_means = {s: np.nanmean([fit_nb_mle(raw_co[s][:, gi])[2] for gi in range(N_GENES)])
            for s in ["SERGIO", "qSimCells", "scMultiSim"]}
ax.barh(list(nb_means.keys()), list(nb_means.values()),
        color=[COLORS_SIM[s] for s in nb_means], alpha=0.85)
ax.set_xlabel("Mean NB log-lik (higher=better)")
ax.set_title("NB goodness of fit")

# Row 2: GRN importance heatmaps
for ci, sim_name in enumerate(["SERGIO", "qSimCells", "scMultiSim"]):
    ax = fig.add_subplot(gs[2, ci])
    sns.heatmap(score_mats[sim_name], ax=ax, cmap="Blues",
                xticklabels=GENE_NAMES, yticklabels=GENE_NAMES,
                annot=True, fmt=".2f", cbar=False, linewidths=0.4)
    for (r, t) in GT_EDGES:
        ax.add_patch(plt.Rectangle((t, r), 1, 1, fill=False, edgecolor="red", lw=2.5))
    ax.set_title(f"{sim_name}\nGENIE3 (red=GT edge)", fontsize=10)

# Row 2 col 3: AUROC bar
ax = fig.add_subplot(gs[2, 3])
auc_sims = ["SERGIO", "qSimCells", "scMultiSim"]
ax.barh(auc_sims, [auroc_results[s] for s in auc_sims],
        color=[COLORS_SIM[s] for s in auc_sims], alpha=0.85)
ax.axvline(0.5, color="k", ls="--", lw=1.2)
ax.set_xlabel("AUROC"); ax.set_title("GRN recovery (GENIE3)")
ax.set_xlim(0, 1)

plt.suptitle("Simulator Benchmark: SERGIO  ·  qSimCells  ·  scMultiSim\n"
             "5-gene cascade, 2 cell types, N=500 cells/type",
             fontsize=14, fontweight="bold", y=1.01)

plt.savefig("benchmark_master_figure.pdf", bbox_inches="tight")
plt.show()
print("Master figure saved.")

## 15  Interpretation

**What to expect from each simulator:**

| Metric | SERGIO | qSimCells | scMultiSim |
|--------|--------|-----------|------------|
| NB fit | Good (Poisson technical noise) | **Best** (NB built in by design) | Moderate (kinetic model) |
| GRN recovery | **Best** (ODE dynamics encode GRN directly) | Good (CX cascade) | Good (CIF-GRN link) |
| Cell-type separation | Good | Good | Good |
| Zero inflation | Moderate | Controlled by Ry angles | Variable |
| Biological realism | Dynamical equilibrium | **Transcriptomic stochasticity** | Spatial/hierarchical |

**Key argument for qSimCells:**  
The quantum Born-rule sampling naturally produces a probability mass function over gene
activation states that is *directly parameterised by the biology* (Ry angles = activation
probabilities; CX gates = regulatory dependencies).  The NB augmentation then adds
realistic count statistics on top of the correct zero-inflation and cascade-correlation
structure — making qSimCells uniquely suited to *transcriptomic stochasticity* simulation
without relying on ODE equilibrium assumptions or spatial proximity proxies.